In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Hello World: Twój pierwszy Circuit kwantowy

Zbuduj [stan Bella](https://en.wikipedia.org/wiki/Bell_state) (dwa splątane Qubit) i uruchom go na trzy sposoby:

1. **Symulacja idealna** — doskonałe wyniki, konto nie jest wymagane
2. **Symulacja z szumem** — symuluje prawdziwe urządzenie, konto nie jest wymagane
3. **Prawdziwy sprzęt kwantowy** — wymaga [konta IBM Quantum](https://janlahmann.github.io/Qiskit-documentation/#setting-up-your-ibm-quantum-account)

## Zbuduj Circuit

In [ ]:
from qiskit import QuantumCircuit

qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)
qc.measure_all()

qc.draw(output="mpl")

## Opcja 1: Symulacja idealna (konto nie jest wymagane)
Używa `StatevectorSampler` — lokalnego symulatora dającego doskonałe, wolne od szumu wyniki.

In [ ]:
from qiskit.primitives import StatevectorSampler

result = StatevectorSampler().run([qc], shots=1024).result()
counts = result[0].data.meas.get_counts()
counts

In [ ]:
from qiskit.visualization import plot_histogram
plot_histogram(counts)

## Opcja 2: Symulacja z szumem (konto nie jest wymagane)
Używa `FakeManilaV2` — lokalnego symulatora odwzorowującego prawdziwe urządzenie kwantowe IBM, włącznie z jego charakterystyką szumu. Circuit musi najpierw zostać poddany transpilacji (adaptacji) do zestawu bramek i topologii Qubit danego urządzenia.

In [ ]:
from qiskit_ibm_runtime import SamplerV2
from qiskit_ibm_runtime.fake_provider import FakeManilaV2
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

backend = FakeManilaV2()
pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
isa_qc = pm.run(qc)

result = SamplerV2(mode=backend).run([isa_qc], shots=1024).result()
counts = result[0].data.meas.get_counts()
counts

In [ ]:
plot_histogram(counts)

## Opcja 3: Prawdziwy sprzęt kwantowy
Wymaga konta IBM Quantum. Szczegóły znajdziesz w sekcji [Konfiguracja konta IBM Quantum](https://janlahmann.github.io/Qiskit-documentation/#setting-up-your-ibm-quantum-account).

Jeśli nie zapisałeś jeszcze swoich danych uwierzytelniających w tej sesji Binder, najpierw uruchom poniższy kod:

In [ ]:
from qiskit_ibm_runtime import QiskitRuntimeService

QiskitRuntimeService.save_account(
    token="<your-api-key>",
    instance="<your-crn>",
    overwrite=True
)

**Uwaga:** Zadania na prawdziwym sprzęcie mogą zająć trochę czasu w zależności od kolejki. Jeśli komórka wciąż działa, możesz sprawdzić status zadania i zobaczyć wyniki na stronie [quantum.cloud.ibm.com/workloads](https://quantum.cloud.ibm.com/workloads?user=me).

In [ ]:
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)
print(f"Running on {backend.name}")

pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
isa_qc = pm.run(qc)

result = SamplerV2(mode=backend).run([isa_qc], shots=1024).result()
counts = result[0].data.meas.get_counts()
counts

In [ ]:
plot_histogram(counts)

## Co dalej?
- **[Samouczki](https://mybinder.org/v2/gh/JanLahmann/Qiskit-documentation/main?filepath=docs/tutorials)** — przewodniki krok po kroku dotyczące algorytmów, mitygacji błędów, transpilacji i nie tylko
- **[Kursy](https://mybinder.org/v2/gh/JanLahmann/Qiskit-documentation/main?filepath=learning/courses)** — ustrukturyzowane ścieżki nauki od podstaw kwantowych po obliczenia w skali użytkowej
- **[Lokalny tryb testowy](https://janlahmann.github.io/Qiskit-documentation/#no-token-use-local-testing-mode)** — uruchamiaj większość notatników bez konta IBM Quantum